In [ ]:
import pandas as pd

df = pd.read_csv('..\\files\\SeaPlaneTravel.csv',parse_dates=['Month'])
df.set_index('Month',inplace=True)
display(df.head())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(18,5))
plt.plot(df)
plt.xlabel('Year')
plt.ylabel('No. of Passengers')
plt.grid(True)
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

def test_stationary(timeseries):

    rol_mean = timeseries.rolling(window=52,center=False).mean()
    rol_std = timeseries.rolling(window=52,center=False).std()

    plt.figure(figsize=(18,5))
    plt.grid(True)
    plt.plot(timeseries)
    plt.plot(rol_mean)
    plt.plot(rol_std)
    plt.show()

    dftest = adfuller(timeseries)
    dfOutput = pd.Series(dftest[0:4],index=['Test Statistics','p-value','#Lags Used','Number of Observations Used'])
    print(dfOutput)
    for key,value in dftest[4].items():
        print(f'Critical Value {key} : {value}')

test_stationary(df)

In [ ]:
import numpy as np

df_log = np.log(df)
test_stationary(df_log)

In [ ]:
df_log_diff = df_log.diff(2).dropna()
test_stationary(df_log_diff)

In [ ]:
from statsmodels.tsa.stattools import acf,pacf

plt.figure(figsize=(18,5))

plt.subplot(121)

plt.title("Auto Correlation Function")
plt.plot(acf(df_log_diff, nlags=12))
plt.axhline(y=0,linestyle='--',color='gray')
plt.axhline(y=-1.96/np.sqrt(len(df_log_diff)),linestyle='--',color='gray')
plt.axhline(y=1.96/np.sqrt(len(df_log_diff)),linestyle='--',color='gray')
plt.axhline(y=0,linestyle='--',color='gray')


plt.subplot(122)
plt.title("Partial Auto Correlation Function")
plt.plot(pacf(df_log_diff,nlags=8))
plt.axhline(y=0,linestyle='--',color='gray')
plt.axhline(y=-1.96/np.sqrt(len(df_log_diff)),linestyle='--',color='gray')
plt.axhline(y=1.96/np.sqrt(len(df_log_diff)),linestyle='--',color='gray')
plt.axhline(y=0,linestyle='--',color='gray')

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarimax_order = (1, 1, 1)
sarimax_seasonal_order = (0, 1, 0, 12)

df_log = df_log.asfreq("MS")

model = SARIMAX(df_log,order=sarimax_order,seasonal_order=sarimax_seasonal_order)
results = model.fit()

print(results.summary())

residual = results.resid.dropna()
rss = np.sum(residual**2)

results.plot_diagnostics(figsize=(15, 7))
plt.show()

In [ ]:
forecast_period = 24
forecast_obj_sarimax = results.get_forecast(forecast_period)
forecast_log_sarimax = forecast_obj_sarimax.predicted_mean
forecast_sarimax = np.exp(forecast_log_sarimax)
display(forecast_sarimax.head())

conf_int_sarimax_log = forecast_obj_sarimax.conf_int()
conf_int_sarimax = np.exp(conf_int_sarimax_log)
display(conf_int_sarimax.head())

In [ ]:
plt.figure(figsize=(18, 8))
plt.plot(df.index, df['#Passengers'], label='Original Data', color='blue')
plt.plot(forecast_sarimax.index, forecast_sarimax, label='SARIMAX Forecast', color='red', linestyle='--')
plt.fill_between(conf_int_sarimax.index,conf_int_sarimax.iloc[:, 0],conf_int_sarimax.iloc[:, 1],color='pink', alpha=0.3, label='Confidence Interval')
plt.title('Sea Plane Passenger Traffic Forecast (SARIMAX)')
plt.xlabel('Month')
plt.ylabel('Number of Passengers')
plt.legend()
plt.grid(True)
plt.show()

In [174]:
import joblib
package = {"model_results": results, "historical_data": df}
joblib.dump(package, "..\\Projects\\models\\seaplane_sarimax_model.joblib")

['..\\Projects\\models\\seaplane_sarimax_model.joblib']